# Advantage Intelligence stroke data

One row per stroke, 46 vendor columns, produced from the results JSON by
`scripts/splitstep_json_to_csv.py`. Regenerate with:

```
python3 scripts/splitstep_json_to_csv.py tests/fixtures/splitstep/*.json -o analysis/data
```

The script empties the vendor's sentinels (`-9999`, `"None"`, `"nan-nan"`) on the way
out, so nothing here needs a `replace(-9999, np.nan)` first. It does **not** touch
coordinates that are finite but physically impossible — those are real signal about
tracking quality, and they get measured below rather than hidden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)

df = pd.read_csv('data/all-matches.csv')
df.shape

## Schema, and three things that will bite

**1. `in` is a Python keyword.** `df.in` is a syntax error. Use `df['in']`.

**2. Six columns are entirely empty, by design.** `rally_id`, `rally_stroke_number`,
`player_id`, `point_score`, `game_score`, `set_score` are the vendor's ground-truth
fields, populated only "from input rally metadata, if provided" — and the job request
has no rally-metadata parameter, so they are never provided. Every analysis builds on
the `pred_*` equivalents, which the vendor flags as Beta. See
`docs/splitstep-integration-spec.md` §4.1.

**3. `pred_player_id` is `Player A` / `Player B`, not a person.** Which one is the
athlete who uploaded the video is decided elsewhere, from wizard input. Nothing in
this file tells you.

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(1),
    'distinct': df.nunique(dropna=True),
})
summary

In [ ]:
# The all-null six. Confirming rather than assuming, because if a future export
# ever does populate them, that changes which columns are authoritative.
empty = df.columns[df.isna().all()].tolist()
print(empty)
df = df.drop(columns=empty)
df.shape

## Court frame

Metres, origin at the **net centre**. `+y` runs toward the far baseline, `+x` toward
the right sideline. This is the vendor's frame, not the database's — `shots.contact_x/y`
measure y from a baseline instead, and `src/lib/services/splitstep/derivation/court.ts`
is the only place that conversion is allowed to happen.

Landmarks below are from that module.

In [ ]:
SINGLES_HALF_WIDTH_M = 4.115
DOUBLES_HALF_WIDTH_M = 5.485
SERVICE_LINE_M = 6.4
BASELINE_M = 11.885

# The playing enclosure: ITF recommended run-off, 6.40 m behind each baseline and
# 3.66 m outside each doubles sideline. Past that you are through the fence, so it is
# a tracking failure rather than a wide ball or a deep returner.
MAX_PLAUSIBLE_X_M = DOUBLES_HALF_WIDTH_M + 3.66   # 9.145
MAX_PLAUSIBLE_Y_M = BASELINE_M + 6.4              # 18.285

def plausible(x, y):
    """Boolean mask: both coordinates present and inside the fence."""
    return (df[x].abs() <= MAX_PLAUSIBLE_X_M) & (df[y].abs() <= MAX_PLAUSIBLE_Y_M)

### How much of the tracking is usable

Positions are nulled as a **pair** downstream — half a position is worse than none,
since a valid x with a corrupt y puts a bounce at a real sideline and a nonsense depth,
which reads as a perfectly plausible shot on a court diagram. Same rule here.

In [ ]:
quality = pd.DataFrame({
    'player':   plausible('player_x_m', 'player_y_m'),
    'opponent': plausible('opponent_x_m', 'opponent_y_m'),
    'bounce':   plausible('bounce_x_m', 'bounce_y_m'),
})

# Percent of strokes whose positions are present and inside the fence.
quality.groupby(df['video_id']).mean().mul(100).round(1)

In [ ]:
# What the implausible ones actually look like. If the tail is mild the bound is
# arguably too tight; if it runs to hundreds of metres it is a tracker failure.
off = df.loc[~quality['bounce'] & df['bounce_y_m'].notna(), ['video_id', 'bounce_x_m', 'bounce_y_m']]
print(f'{len(off)} bounces outside the enclosure')
off['bounce_y_m'].abs().describe()

## Rallies

Group by `pred_rally_id`, order by `pred_rally_stroke_number`. `pred_rally_id` is only
unique *within* a video, so any cross-match grouping needs both keys.

In [ ]:
df = df.sort_values(['video_id', 'pred_rally_id', 'pred_rally_stroke_number'])
rallies = df.groupby(['video_id', 'pred_rally_id'])

rally_stats = pd.DataFrame({
    'strokes': rallies.size(),
    'server': rallies['pred_player_id'].first(),
    'serves': rallies['stroke_type'].apply(lambda s: (s == 'serve').sum()),
    'ended_in': rallies['in'].last(),
    'ended_net': rallies['net_hit'].last(),
    'duration_s': rallies['time'].max() - rallies['time'].min(),
})
rally_stats.head(10)

In [ ]:
# Rally length distribution, per match.
fig, ax = plt.subplots(figsize=(9, 4))
for video, group in rally_stats.groupby(level='video_id'):
    ax.hist(group['strokes'], bins=np.arange(0.5, 25.5), alpha=0.55, label=video)
ax.set_xlabel('strokes in rally'); ax.set_ylabel('rallies'); ax.legend()
ax.set_title('Rally length')
plt.show()

rally_stats.groupby(level='video_id')['strokes'].describe()

## Serves

Two consecutive `serve` strokes by the same player in one rally means the first was a
fault. Whether faulted serves are emitted at all is **open question 1** in the spec —
unconfirmed with the vendor — so treat the first/second split as a hypothesis to test
against these counts, not as fact.

In [ ]:
serves = df[df['stroke_type'] == 'serve'].copy()
serves['serve_number'] = serves.groupby(['video_id', 'pred_rally_id']).cumcount() + 1

serves.groupby(['video_id', 'pred_player_id', 'serve_number']).agg(
    n=('speed_kmh', 'size'),
    mean_kmh=('speed_kmh', 'mean'),
    max_kmh=('speed_kmh', 'max'),
    in_pct=('in', lambda s: s.mean() * 100),
).round(1)

In [ ]:
# Serve placement, plotted in the vendor's frame. Only the bounces that survive the
# plausibility mask, or the outliers rescale the axes into uselessness.
ok = serves.loc[plausible('bounce_x_m', 'bounce_y_m').reindex(serves.index, fill_value=False)]

fig, ax = plt.subplots(figsize=(6, 8))
for player, group in ok.groupby('pred_player_id'):
    ax.scatter(group['bounce_x_m'], group['bounce_y_m'], s=14, alpha=0.6, label=player)

for x in (-SINGLES_HALF_WIDTH_M, 0, SINGLES_HALF_WIDTH_M):
    ax.axvline(x, color='0.7', lw=0.8)
for y in (-BASELINE_M, -SERVICE_LINE_M, 0, SERVICE_LINE_M, BASELINE_M):
    ax.axhline(y, color='0.7', lw=0.8)
ax.axhline(0, color='0.3', lw=1.4)  # net

ax.set_aspect('equal'); ax.set_xlabel('x (m, + = right sideline)')
ax.set_ylabel('y (m, 0 = net)'); ax.legend(); ax.set_title('Serve bounce locations')
plt.show()

## Strokes

`stroke_score` and `side_score` are the model's confidence in `stroke_type` and
`stroke_side`. Both floor near 0.5 — a coin flip — but the low-confidence tail is
small: roughly 1% of strokes sit below 0.9. So a confidence cut is cheap here, and
worth applying to any stroke-side split rather than taking every label at face
value. Check the cost on your own export before assuming it stays that cheap.

In [ ]:
df.groupby(['stroke_type', 'stroke_side']).agg(
    n=('speed_kmh', 'size'),
    mean_kmh=('speed_kmh', 'mean'),
    mean_rpm=('ang_vel_mag_rpm', 'mean'),
    in_pct=('in', lambda s: s.mean() * 100),
    mean_conf=('side_score', 'mean'),
).round(1)

In [ ]:
# How much data a confidence cut costs.
for cut in (0.0, 0.6, 0.7, 0.8, 0.9):
    kept = (df['side_score'] >= cut).mean() * 100
    print(f'side_score >= {cut:.1f}  keeps {kept:5.1f}% of strokes')